In [ ]:
import numpy as np
from os import listdir
from tqdm.auto import tqdm

from scipy.interpolate import CubicSpline

from threading import Thread

from pendulibrary.integrate import integrate_state
from pendulibrary.common_targetters import closest_sol
from pendulibrary.targeter import dc_underconstrained

In [ ]:
def preprop_one(fname, frac, indx, data_out: dict):
    data = np.load(f"../database/{fname}.npz")
    x0s = data["x0s"]
    periods = data["periods"]
    Lr, Mr = data["params"]
    # tangents = data["tangents"]
    cols = np.column_stack((x0s, periods))
    steps = np.linalg.norm(np.diff(cols, axis=0), axis=1)
    Ss = np.append(0, np.cumsum(steps))

    spline_x = CubicSpline(Ss, x0s)
    spline_T = CubicSpline(Ss, periods)

    targ = closest_sol(Lr, Mr, 1e-14)

    Xg = spline_x(frac * Ss[-1])
    Tg = spline_T(frac * Ss[-1])

    ind = np.argmin(np.abs(frac * Ss[-1] - Ss))

    Xs = np.hstack((x0s, periods[:, None]))
    if ind == 0:
        ds = np.linalg.norm(Xs[0] - Xs[1])
    elif ind == Xs.shape[0] - 1:
        ds = np.linalg.norm(Xs[-1] - Xs[-2])
    else:
        ds1 = np.linalg.norm(Xs[ind] - Xs[ind - 1])
        ds2 = np.linalg.norm(Xs[ind + 1] - Xs[ind])
        ds = max(ds1, ds2)

    XX = np.append(Xg, Tg)
    try:
        X, _, _ = dc_underconstrained(
            XX, targ.g_dg_stm, 1e-6, max_step=ds, max_iter=100
        )
    except RuntimeError:
        print(f"Error on {fname}-{frac}")
        return 0
    x0 = X[:-1]
    period = X[-1]

    ts, xs, fs = integrate_state(x0, period, Lr, Mr, 1e-14)

    data_out[indx] = {
        "t": ts.copy(),
        "thetas": xs[:, :2].copy(),
        "omegas": fs[:, :2].copy(),
    }
    # return ts, xs, fs


def preprop_fam(fam, fracs, data_all: dict):
    data_all[fam] = dict()

    procs = []
    for indx, frac in enumerate(fracs):
        p = Thread(target=preprop_one, args=(fam, frac, indx, data_all[fam]))
        procs.append(p)

    for proc in procs:
        proc.start()
    for proc in procs:
        proc.join()

In [ ]:
fams = [f.removesuffix(".npz") for f in listdir("../database")]
fracs = np.linspace(1, 0, 25, False)[::-1]

In [ ]:
all_data = {}
for fam in tqdm(fams):
    preprop_fam(fam, fracs, all_data)

In [ ]:
import pickle

with open('data.pkl', 'wb') as f:
    pickle.dump(all_data, f)